# Ampliación del dataset para modelo v5 — extracción y pre-anotación

Este cuaderno prepara imágenes nuevas para ampliar el dataset del detector (de v4 a v5), usando vídeos del **split train** de SoccerNet-Tracking (clips DISTINTOS de los de test, para no contaminar la evaluación).

**Flujo de este cuaderno (parte automática):**
1. Descargar el split train de tracking (57 clips).
2. Extraer ~2000 frames espaciados (variados, no consecutivos).
3. Aplicar el modelo v4 para pre-anotar las cajas (formato YOLO).
4. Empaquetar imágenes + pre-anotaciones para subir a Roboflow.

**Nota anti-fuga de datos:** los frames salen del split TRAIN, nunca del test (SNMOT-192 y los demás clips de test quedan reservados solo para evaluar). Entrenar con train y evaluar con test es la práctica correcta y evita el overfitting al conjunto de test.

In [1]:
# ---- CELDA 1: Descargar el split TRAIN de tracking (57 clips, ~varios GB) ----
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "SoccerNet", "--quiet"])

from SoccerNet.Downloader import SoccerNetDownloader
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
SOCCERNET_DIR = REPO_ROOT / "data" / "soccernet"

downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))
downloader.password = "TU_CONTRASEÑA_NDA_AQUI"   # <--- tu contraseña NDA

# Descargar SOLO el split train (los clips de test NO se tocan: son para evaluar)
downloader.downloadDataTask(task="tracking", split=["train"])
print("Descarga del split train completada.")

# Descomprimir si vino en zip (como pasó con el test)
import zipfile
zip_train = SOCCERNET_DIR / "tracking" / "train.zip"
destino = SOCCERNET_DIR / "tracking" / "train"
if zip_train.exists() and not destino.exists():
    print(f"Descomprimiendo {zip_train.name}...")
    with zipfile.ZipFile(zip_train, "r") as z:
        z.extractall(SOCCERNET_DIR / "tracking")
    print("Descomprimido.")

# Localizar los clips de train (ajustar si hay doble anidamiento train/train)
TRAIN_DIR = SOCCERNET_DIR / "tracking" / "train"
if (TRAIN_DIR / "train").exists():
    TRAIN_DIR = TRAIN_DIR / "train"
clips = sorted([d for d in TRAIN_DIR.iterdir() if d.is_dir() and d.name.startswith("SNMOT")])
print(f"Clips de train disponibles: {len(clips)}")
print(f"Primeros: {[c.name for c in clips[:5]]}")


C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descarga del split train completada.
Descomprimiendo train.zip...
Descomprimido.
Clips de train disponibles: 57
Primeros: ['SNMOT-060', 'SNMOT-061', 'SNMOT-062', 'SNMOT-063', 'SNMOT-064']


In [2]:
# ---- CELDA 2: Extraer ~2000 frames ESPACIADOS de los clips de train ----
import cv2, random
from pathlib import Path

FRAMES_DIR = REPO_ROOT / "data" / "dataset_v5" / "images"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

OBJETIVO_TOTAL = 2000
ESPACIADO = 20   # tomar 1 frame de cada 20 (evita frames casi idénticos)

# Repartir el objetivo entre los clips disponibles
frames_por_clip = max(1, OBJETIVO_TOTAL // len(clips))
print(f"Objetivo: ~{OBJETIVO_TOTAL} frames | {frames_por_clip} por clip | espaciado {ESPACIADO}")

total = 0
for clip in clips:
    img_dir = clip / "img1"
    jpgs = sorted(img_dir.glob("*.jpg"))
    if not jpgs: continue
    # tomar frames espaciados, hasta frames_por_clip por clip
    elegidos = jpgs[::ESPACIADO][:frames_por_clip]
    for src in elegidos:
        img = cv2.imread(str(src))
        if img is None: continue
        # nombre único: clip + nombre de frame original
        out_name = f"{clip.name}_{src.stem}.jpg"
        cv2.imwrite(str(FRAMES_DIR / out_name), img)
        total += 1

print(f"\nFrames extraídos: {total}")
print(f"Guardados en: {FRAMES_DIR}")


Objetivo: ~2000 frames | 35 por clip | espaciado 20

Frames extraídos: 1995
Guardados en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v5\images


In [3]:
# ---- CELDA 3: Aplicar el modelo v4 para PRE-ANOTAR (formato YOLO) ----
from ultralytics import YOLO
from pathlib import Path

MODELO_V4 = REPO_ROOT / "models" / "best_v4.pt"   # ajusta la ruta si tu v4 está en otro sitio
detector = YOLO(str(MODELO_V4))
print(f"Modelo v4 cargado. Clases: {detector.names}")

LABELS_DIR = REPO_ROOT / "data" / "dataset_v5" / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)

jpgs = sorted(FRAMES_DIR.glob("*.jpg"))
print(f"Pre-anotando {len(jpgs)} frames...")

CONF_PREANOT = 0.25   # confianza para pre-anotar (luego corriges a mano en Roboflow)
n_cajas = 0
for i, img_path in enumerate(jpgs):
    res = detector(str(img_path), conf=CONF_PREANOT, imgsz=1280, verbose=False)[0]
    # formato YOLO: clase x_centro y_centro ancho alto (normalizado 0-1)
    h, w = res.orig_shape
    lineas = []
    if res.boxes is not None:
        for box, cls in zip(res.boxes.xywhn.cpu().numpy(), res.boxes.cls.cpu().numpy().astype(int)):
            xc, yc, bw, bh = box
            lineas.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            n_cajas += 1
    # guardar el .txt con el mismo nombre que la imagen
    (LABELS_DIR / f"{img_path.stem}.txt").write_text("\n".join(lineas), encoding="utf-8")
    if (i+1) % 200 == 0:
        print(f"  {i+1}/{len(jpgs)} frames pre-anotados...")

print(f"\nPre-anotación completa: {len(jpgs)} frames, {n_cajas} cajas")
print(f"Labels en: {LABELS_DIR}")
print(f"Media: {n_cajas/max(len(jpgs),1):.1f} cajas/frame")


Modelo v4 cargado. Clases: {0: 'ball', 1: 'player', 2: 'referee'}
Pre-anotando 1995 frames...
  200/1995 frames pre-anotados...
  400/1995 frames pre-anotados...
  600/1995 frames pre-anotados...
  800/1995 frames pre-anotados...
  1000/1995 frames pre-anotados...
  1200/1995 frames pre-anotados...
  1400/1995 frames pre-anotados...
  1600/1995 frames pre-anotados...
  1800/1995 frames pre-anotados...

Pre-anotación completa: 1995 frames, 30474 cajas
Labels en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v5\labels
Media: 15.3 cajas/frame


## Siguiente paso: subir a Roboflow y corregir (MANUAL)

Ya tienes en `data/dataset_v5/`:
- `images/` → los ~2000 frames extraídos
- `labels/` → las pre-anotaciones del modelo v4 en formato YOLO

**En Roboflow (interfaz web), haz esto:**

1. **Crea/abre tu proyecto** del detector de fútbol (el mismo donde está el v4, para reutilizar las clases: 0=ball, 1=player, 2=referee).
2. **Sube las imágenes con sus labels**: arrastra la carpeta `images/` y Roboflow detectará los `.txt` de `labels/` si los subes juntos (o usa la opción de subir imágenes + anotaciones YOLO). Las pre-anotaciones del v4 aparecerán como cajas ya puestas.
3. **CORRIGE las cajas a mano** (el paso costoso): revisa cada frame, ajusta cajas mal puestas, añade jugadores que el v4 no detectó, borra falsos positivos. Presta atención a jugadores lejanos/ocluidos (donde el v4 falla más, como viste en la evaluación).
4. **Añade estas imágenes a tu dataset existente** (las 750 que ya tenías), para tener 750 + 2000 = 2750.
5. **Haz el split** train/valid/test (por ejemplo 70/20/10).
6. **Exporta en formato YOLOv8** (sin augmentation de Roboflow — el augmentation lo harás tú en el cuaderno aparte, solo sobre el train).

**Importante:** al exportar, NO actives el augmentation de Roboflow. Exporta el dataset "limpio" (solo las imágenes reales con sus cajas corregidas). El augmentation realista (espejo, brillo, ruido) se aplica después, en el cuaderno de augmentation, y SOLO sobre el conjunto de train.

Cuando tengas el dataset exportado de Roboflow, pasamos al cuaderno de augmentation.

In [1]:
from pathlib import Path
import zipfile, collections, yaml

ZIP  = Path(r"C:\Users\sgghu\Downloads\football-tactical-analysis.yolov8.zip")
DEST = Path(r"C:\Users\sgghu\Downloads\football-tactical-analysis")   # destino al descomprimir

with zipfile.ZipFile(ZIP) as z:
    z.extractall(DEST)
print("Descomprimido en:", DEST, "\n")

d = yaml.safe_load(open(DEST/"data.yaml", encoding="utf-8"))
print("CLASES (orden = índice):", d["names"], "| nc =", d["nc"], "\n")

def split_dir(root, canon):
    for c in ([canon, "val"] if canon == "valid" else [canon]):
        if (root/c/"images").exists(): return root/c
    return None

gran_total = 0
for canon in ("train", "valid", "test"):
    src = split_dir(DEST, canon)
    if src is None: continue
    cont = collections.Counter()
    for img in (src/"images").iterdir():
        if img.suffix.lower() not in (".jpg", ".jpeg", ".png"): continue
        cont[img.name.split("_")[0]] += 1     # secuencia = primer token (SNMOT-061, etc.)
    tot = sum(cont.values()); gran_total += tot
    print(f"[{canon}]  total={tot}")
    for pref, n in sorted(cont.items()):
        print(f"   {pref}: {n}")
    print()
print("TOTAL imágenes:", gran_total)

Descomprimido en: C:\Users\sgghu\Downloads\football-tactical-analysis 

CLASES (orden = índice): ['ball', 'player', 'referee'] | nc = 3 

[train]  total=1911
   A1606b0e6: 372
   C35bd9041: 91
   E3c993bd2: 57
   SNMOT-060: 29
   SNMOT-061: 24
   SNMOT-062: 23
   SNMOT-063: 18
   SNMOT-064: 20
   SNMOT-065: 24
   SNMOT-066: 29
   SNMOT-067: 26
   SNMOT-068: 25
   SNMOT-069: 24
   SNMOT-070: 25
   SNMOT-071: 23
   SNMOT-072: 26
   SNMOT-073: 22
   SNMOT-074: 26
   SNMOT-075: 20
   SNMOT-076: 25
   SNMOT-077: 28
   SNMOT-097: 25
   SNMOT-098: 23
   SNMOT-099: 29
   SNMOT-100: 25
   SNMOT-101: 22
   SNMOT-102: 25
   SNMOT-103: 21
   SNMOT-104: 22
   SNMOT-105: 24
   SNMOT-106: 23
   SNMOT-107: 28
   SNMOT-108: 24
   SNMOT-109: 21
   SNMOT-110: 28
   SNMOT-111: 23
   SNMOT-112: 27
   SNMOT-113: 22
   SNMOT-114: 26
   SNMOT-115: 26
   SNMOT-151: 26
   SNMOT-152: 24
   SNMOT-153: 28
   SNMOT-154: 31
   SNMOT-155: 26
   SNMOT-156: 21
   SNMOT-157: 27
   SNMOT-158: 21
   SNMOT-159: 24
   SNMOT

In [2]:
from pathlib import Path
import shutil, yaml

DEST = Path(r"C:\Users\sgghu\Downloads\football-tactical-analysis")  # el descomprimido
OUT  = Path(r"C:\Users\sgghu\Downloads\dataset_revisadas")            # solo revisadas

ALLOW = {f"SNMOT-{n:03d}" for n in range(60, 73)}   # SNMOT-060 ... SNMOT-072

def split_dir(root, canon):
    for c in ([canon, "val"] if canon == "valid" else [canon]):
        if (root/c/"images").exists(): return root/c
    return None

total = 0; vistos = set()
for canon in ("train", "valid", "test"):
    src = split_dir(DEST, canon)
    if src is None: continue
    (OUT/canon/"images").mkdir(parents=True, exist_ok=True)
    (OUT/canon/"labels").mkdir(parents=True, exist_ok=True)
    n = 0
    for img in (src/"images").iterdir():
        if img.name.split("_")[0] not in ALLOW: continue
        vistos.add(img.name.split("_")[0])
        shutil.copy(img, OUT/canon/"images"/img.name)
        lbl = src/"labels"/(img.stem + ".txt")
        if lbl.exists(): shutil.copy(lbl, OUT/canon/"labels"/lbl.name)
        n += 1
    print(f"{canon}: {n}"); total += n
print("TOTAL revisadas:", total, "(esperado ~455)")
print("Secuencias incluidas:", sorted(vistos))

d = yaml.safe_load(open(DEST/"data.yaml", encoding="utf-8"))
yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "valid/images",
                "test": "test/images", "nc": d["nc"], "names": d["names"]},
               open(OUT/"data.yaml", "w", encoding="utf-8"), sort_keys=False, allow_unicode=True)
print("data.yaml en", OUT)

train: 316
valid: 98
test: 41
TOTAL revisadas: 455 (esperado ~455)
Secuencias incluidas: ['SNMOT-060', 'SNMOT-061', 'SNMOT-062', 'SNMOT-063', 'SNMOT-064', 'SNMOT-065', 'SNMOT-066', 'SNMOT-067', 'SNMOT-068', 'SNMOT-069', 'SNMOT-070', 'SNMOT-071', 'SNMOT-072']
data.yaml en C:\Users\sgghu\Downloads\dataset_revisadas


In [4]:
from roboflow import Roboflow
from getpass import getpass
from pathlib import Path

V2 = Path(r"C:\Users\sgghu\Downloads\dataset_v4_v2")   # = dataset que entrenó v4 (versión 2)
rf = Roboflow(api_key=getpass("API key de Roboflow: "))
proj = rf.workspace("hugos-workspace-0dwnk").project("football-tactical-analysis")
proj.version(2).download("yolov8", location=str(V2))
print("v4 dataset (versión 2) en", V2)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to C:\Users\sgghu\Downloads\dataset_v4_v2 in yolov8:: 100%|██████████| 1491/1491 [00:00<00:00, 1594.12it/s]


v4 dataset (versión 2) en C:\Users\sgghu\Downloads\dataset_v4_v2


In [5]:
from pathlib import Path
import shutil, yaml, collections

V2    = Path(r"C:\Users\sgghu\Downloads\dataset_v4_v2")       # v4 data (versión 2)
REVIS = Path(r"C:\Users\sgghu\Downloads\dataset_revisadas")    # 455 nuevas revisadas
OUT   = Path(r"C:\Users\sgghu\Downloads\dataset_v5")           # combinado para v5

EXCLUIR = ()   # vacío = dejas A1606 (consistente con v4). Para demo 100% no visto: ("A1606",)

def names_de(r):
    nm = yaml.safe_load(open(r/"data.yaml", encoding="utf-8"))["names"]
    return [n.lower() for n in (nm if isinstance(nm, list) else [nm[k] for k in sorted(nm, key=int)])]

n2, nr = names_de(V2), names_de(REVIS)
assert set(n2) == set(nr), f"clases distintas: {n2} vs {nr}"
remap = {i: n2.index(n) for i, n in enumerate(nr)}

def split_dir(root, canon):
    for c in ([canon, "val"] if canon == "valid" else [canon]):
        if (root/c/"images").exists(): return root/c
    return None

comp = collections.Counter()
for root, pref, rm in [(V2, "v4", None), (REVIS, "rev", remap)]:
    for canon in ("train", "valid", "test"):
        src = split_dir(root, canon)
        if src is None: continue
        (OUT/canon/"images").mkdir(parents=True, exist_ok=True)
        (OUT/canon/"labels").mkdir(parents=True, exist_ok=True)
        for img in (src/"images").iterdir():
            if img.suffix.lower() not in (".jpg", ".jpeg", ".png"): continue
            if any(img.name.startswith(p) for p in EXCLUIR):
                comp[("excluida", img.name.split("_")[0])] += 1; continue
            shutil.copy(img, OUT/canon/"images"/f"{pref}_{img.name}")
            lbl = src/"labels"/(img.stem + ".txt")
            ol = OUT/canon/"labels"/f"{pref}_{img.stem}.txt"
            if lbl.exists():
                fil = []
                for ln in open(lbl, encoding="utf-8"):
                    p = ln.split()
                    if not p: continue
                    c = rm[int(p[0])] if rm else int(p[0])
                    fil.append(" ".join([str(c), *p[1:]]))
                ol.write_text("\n".join(fil) + ("\n" if fil else ""), encoding="utf-8")
            else:
                ol.write_text("", encoding="utf-8")
            comp[(canon, pref)] += 1

print("Composición v5:")
for k, n in sorted(comp.items()): print(f"  {k}: {n}")
yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "valid/images",
                "test": "test/images", "nc": len(n2), "names": n2},
               open(OUT/"data.yaml", "w", encoding="utf-8"), sort_keys=False, allow_unicode=True)
print("data.yaml v5 en", OUT)

Composición v5:
  ('test', 'rev'): 41
  ('test', 'v4'): 74
  ('train', 'rev'): 316
  ('train', 'v4'): 520
  ('valid', 'rev'): 98
  ('valid', 'v4'): 149
data.yaml v5 en C:\Users\sgghu\Downloads\dataset_v5


In [6]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")   # mismo arranque que v4; o "best_v4.pt" para continuar desde él
results = model.train(
    data=r"C:\Users\sgghu\Downloads\dataset_v5\data.yaml",
    epochs=50, imgsz=1280, batch=2, device=0,
    patience=15, workers=0, cache=False, amp=True,
    project=r"C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\runs\detect\models",
    name="detector_v5_1280",
)

New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sgghu\Downloads\dataset_v5\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0,